# Auto-encodeurs débruiteurs et variationnels avec PyTorch Lightning

Le notebook précédent entraînait des auto-encodeurs à reconstruire leur entrée. Celui-ci change d'abord la tâche : retrouver une image propre à partir d'une version bruitée. Puis il change le modèle, pour pouvoir générer de nouvelles images. Les données sont celles de Fashion-MNIST : 60 000 images d'entraînement et 10 000 de test, 28 × 28 en niveaux de gris comme MNIST, mais de 10 sortes de vêtements et d'accessoires.

1. Corrompre les images
2. Un débruiteur dense
3. Un code plus large
4. Un débruiteur convolutif
5. Un auto-encodeur variationnel
6. Générer des images

Chaque partie réutilise les données et les modèles des précédentes. Pas besoin de GPU : le notebook tourne en 20 à 25 minutes sur un processeur. Un GPU (`Exécution > Modifier le type d'exécution`) l'accélère.

## Installation et imports

In [ ]:
!pip install -q lightning

In [ ]:
from collections import OrderedDict

import lightning
import matplotlib.pyplot as plt
import torch
import torchvision
from lightning.pytorch.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

lightning.seed_everything(0)

## Chargement de Fashion-MNIST

In [ ]:
train_data = torchvision.datasets.FashionMNIST("data", train=True, download=True)
test_data = torchvision.datasets.FashionMNIST("data", train=False, download=True)
class_names = [
  "T-shirt",
  "Pantalon",
  "Pull",
  "Robe",
  "Manteau",
  "Sandale",
  "Chemise",
  "Basket",
  "Sac",
  "Bottine",
]
input_dim = 28 * 28
# Pixels dans [0, 1], comme la sortie d'une sigmoïde
X_train = train_data.data.reshape(-1, input_dim).float() / 255
y_train = train_data.targets
X_test = test_data.data.reshape(-1, input_dim).float() / 255
y_test = test_data.targets
print(X_train.shape, X_test.shape)


def show_images(*rows: torch.Tensor, titles: list | None = None) -> None:
  # Affiche des rangées d'images 28 × 28, une rangée par tableau
  n = len(rows[0])
  _, axes = plt.subplots(
    len(rows), n, figsize=(n * 1.2, len(rows) * 1.3), squeeze=False
  )
  for row, images in zip(axes, rows):
    for ax, image in zip(row, images):
      ax.imshow(image.reshape(28, 28), cmap="gray_r")
      ax.axis("off")
  for ax, title in zip(axes[0], [] if titles is None else titles):
    ax.set_title(title, fontsize=8)
  plt.show()


show_images(X_train[:10], titles=[class_names[y] for y in y_train[:10]])

## 1. Corrompre les images

Un débruiteur apprend à retrouver l'image propre à partir d'une version abîmée : ses exemples d'entraînement sont des paires (image bruitée, image propre). Fabriquez les images bruitées :

- `X_train_noisy` et `X_test_noisy` : `X_train` et `X_test`, plus un bruit gaussien de moyenne 0 et d'écart-type 0,5 tiré indépendamment pour chaque pixel ;
- ramenez ensuite chaque pixel dans [0, 1], comme ceux d'une vraie image.

La cellule après la solution affiche des images bruitées au-dessus des images propres.

**Documentation utile**

- [`torch.randn`](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)
- [`torch.Generator`](https://docs.pytorch.org/docs/stable/generated/torch.Generator.html)
- [`torch.clamp`](https://docs.pytorch.org/docs/stable/generated/torch.clamp.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `generator = torch.Generator().manual_seed(0)`, puis `torch.randn(X_train.shape, generator=generator)` : un bruit de moyenne 0 et d'écart-type 1, de la forme des images, un tirage par pixel ; multiplié par 0,5, son écart-type vaut 0,5
- `X.clamp(0, 1)` : les valeurs sous 0 deviennent 0, celles au-dessus de 1 deviennent 1
- vérification : `X_train_noisy.min()` vaut 0, `X_train_noisy.max()` vaut 1, et `X_train_noisy.shape` est celle de `X_train`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
generator = torch.Generator().manual_seed(0)
X_train_noisy = (X_train + 0.5 * torch.randn(...)).clamp(..., ...)
X_test_noisy = ...
```

### Solution

In [ ]:
generator = torch.Generator().manual_seed(0)
X_train_noisy = (X_train + 0.5 * torch.randn(X_train.shape, generator=generator)).clamp(
  0, 1
)
X_test_noisy = (X_test + 0.5 * torch.randn(X_test.shape, generator=generator)).clamp(
  0, 1
)

### Les images bruitées

In [ ]:
titles = [class_names[y] for y in y_test[:10]]
show_images(X_test_noisy[:10], X_test[:10], titles=titles)

## Les outils de l'entraînement

Ils sont fournis, et servent à tout le notebook :

- `AutoEncoder`, un `LightningModule` qui enveloppe un réseau et une perte : son entrée est l'image bruitée, sa cible l'image propre ; il initialise les poids du réseau comme Keras, ce qui évite que des unités ReLU du code meurent ;
- `make_loaders(X_noisy, X, X_test_noisy, X_test)`, qui rend les `DataLoader` d'entraînement et de validation des paires (image bruitée, image propre), par lots de 256 ; la validation se fait sur les images de test ;
- `train(model, loaders, max_epochs, name)`, qui entraîne un `AutoEncoder` et rend sa dernière perte de validation ;
- `predict(network, X)`, qui applique un réseau à toutes les images de `X`, lot par lot, sans calculer de gradient.

In [ ]:
class AutoEncoder(lightning.LightningModule):
  """Lightning wrapper: reconstruct the clean image from the noisy one."""

  def __init__(
    self,
    model: nn.Module,
    loss: nn.Module,
    learning_rate: float = 1e-3,
    input_shape: tuple[int, ...] = (input_dim,),
  ) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model", "loss"])
    self.model = model
    self.loss = loss
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, *input_shape)
    # Initialisation de Glorot, celle de Keras : avec celle de PyTorch, la plupart
    # des unités ReLU d'un petit code meurent (nulles pour toutes les images)
    for layer in model.modules():
      if isinstance(layer, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)

  def forward(self, images: torch.Tensor) -> torch.Tensor:
    return self.model(images)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    noisy_images, images = batch
    loss = self.loss(self(noisy_images), images)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss

  def training_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(
    self, batch: tuple[torch.Tensor, torch.Tensor], batch_index: int
  ) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)


def make_loaders(
  X_noisy: torch.Tensor,
  X: torch.Tensor,
  X_test_noisy: torch.Tensor,
  X_test: torch.Tensor,
) -> tuple[DataLoader, DataLoader]:
  """Training and validation loaders of the pairs (noisy image, clean image)."""
  train_loader = DataLoader(TensorDataset(X_noisy, X), batch_size=256, shuffle=True)
  val_loader = DataLoader(TensorDataset(X_test_noisy, X_test), batch_size=256)
  return train_loader, val_loader


def train(
  model: AutoEncoder,
  loaders: tuple[DataLoader, DataLoader],
  max_epochs: int,
  name: str,
) -> float:
  """Train model and return its last validation loss."""
  trainer = lightning.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name=name),
    # La barre de progression de tqdm, plutôt que celle de rich que Lightning
    # choisit quand rich est installé (Colab) : elle a figé des entraînements
    callbacks=[TQDMProgressBar()],
    enable_checkpointing=False,
    enable_model_summary=False,
  )
  trainer.fit(model, *loaders)
  return trainer.callback_metrics["val_loss"].item()


@torch.no_grad()
def predict(model: nn.Module, X: torch.Tensor, batch_size: int = 256) -> torch.Tensor:
  """Apply a network to every image of X, batch by batch."""
  device = next(model.parameters()).device
  training = model.training
  model.eval()
  predictions = [model(batch.to(device)).cpu() for batch in X.split(batch_size)]
  # Rend au réseau son mode : fit ne le remet plus en mode entraînement
  model.train(training)
  return torch.cat(predictions)


loaders = make_loaders(X_train_noisy, X_train, X_test_noisy, X_test)

## 2. Un débruiteur dense

Construisez `autoencoder`, un débruiteur dense à une couche cachée, et `model`, l'`AutoEncoder` qui l'enveloppe :

- une couche `nn.Linear` de `input_dim` vers `encoding_dim = 12` unités, suivie d'une ReLU : le code ;
- la couche de sortie, de `encoding_dim` vers `input_dim`, suivie d'une activation adaptée à des pixels dans [0, 1] ;
- la perte `nn.MSELoss()`.

Affichez son résumé avec `ModelSummary`. La cellule après la solution l'entraîne sur les paires (image bruitée, image propre), puis affiche des images de test bruitées, leur version débruitée et l'image propre.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- un seul `nn.Sequential` suffit : `nn.Linear`, `nn.ReLU()`, `nn.Linear`, puis l'activation de sortie
- la sortie : `nn.Sigmoid()`, qui rend des valeurs dans ]0, 1[, comme les pixels
- `print(ModelSummary(model, max_depth=-1))` : 9,4 K paramètres pour la couche du code (784 × 12 + 12) et 10,2 K pour la sortie (12 × 784 + 784), 19,6 K en tout

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encoding_dim = 12
autoencoder = nn.Sequential(nn.Linear(..., ...), nn.ReLU(), ..., ...)
model = AutoEncoder(autoencoder, loss=...)
print(ModelSummary(model, max_depth=-1))
```

### Solution

In [ ]:
encoding_dim = 12
autoencoder = nn.Sequential(
  nn.Linear(input_dim, encoding_dim),
  nn.ReLU(),
  nn.Linear(encoding_dim, input_dim),
  nn.Sigmoid(),  # sortie dans ]0, 1[, comme les pixels
)
model = AutoEncoder(autoencoder, loss=nn.MSELoss())
print(ModelSummary(model, max_depth=-1))

### Entraînement

L'entrée est l'image bruitée, la cible l'image propre. En haut, les images bruitées, puis leur version débruitée, puis les images propres.

In [ ]:
val_loss = train(model, loaders, max_epochs=20, name="dense_12")
print(f"Perte de validation (MSE) : {val_loss:.4f}")
X_test_denoised = predict(autoencoder, X_test_noisy)
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

## 3. Un code plus large

Un code de 12 nombres limite ce que le débruiteur peut restituer. Refaites-le avec un code de `encoding_dim = 64` unités, entraînez-le comme dans la partie 2 (le même appel à `train`, sous un autre `name`), et affichez les mêmes images.

La perte de validation baisse-t-elle ? Les images sont-elles plus nettes ? Le réseau recopie-t-il le bruit, maintenant qu'il aurait la place de le faire ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- la solution de la partie 2 avec `encoding_dim = 64`, puis la cellule d'entraînement de la partie 2, copiées l'une après l'autre
- comparez la perte de validation à celle de la partie 2 ; le résumé affiche 101 K paramètres, cinq fois plus

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
encoding_dim = 64
autoencoder = ...
model = AutoEncoder(autoencoder, loss=nn.MSELoss())
val_loss = train(model, loaders, max_epochs=..., name="dense_64")
print(f"Perte de validation (MSE) : {val_loss:.4f}")
X_test_denoised = ...
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)
```

### Solution

In [ ]:
encoding_dim = 64
autoencoder = nn.Sequential(
  nn.Linear(input_dim, encoding_dim),
  nn.ReLU(),
  nn.Linear(encoding_dim, input_dim),
  nn.Sigmoid(),
)
model = AutoEncoder(autoencoder, loss=nn.MSELoss())
val_loss = train(model, loaders, max_epochs=20, name="dense_64")
print(f"Perte de validation (MSE) : {val_loss:.4f}")
X_test_denoised = predict(autoencoder, X_test_noisy)
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

La perte de validation baisse nettement, d'environ 0,025 à 0,018, et les images sont plus nettes : on distingue les cols, les manches et les motifs, comme les carreaux de la chemise. Le réseau ne recopie pas le bruit, alors qu'il aurait maintenant la place de le faire : sa cible est l'image propre, et chaque pixel de bruit recopié augmenterait la perte. C'est ce qui permet à un débruiteur d'avoir un code large sans apprendre l'identité, contrairement à un auto-encodeur ordinaire.

## 4. Un débruiteur convolutif

Une couche dense traite chaque pixel à part ; une convolution exploite le voisinage de chaque pixel, ce qui convient mieux à une image. Les couches de convolution de PyTorch attendent des images 1 × 28 × 28, avec un axe pour les canaux juste après celui du lot (un seul canal en niveaux de gris, trois en couleur) : la cellule suivante remet les images dans cette forme, et prépare les `DataLoader` de ces images.

In [ ]:
X_train_img = X_train.reshape(-1, 1, 28, 28)
X_test_img = X_test.reshape(-1, 1, 28, 28)
X_train_noisy_img = X_train_noisy.reshape(-1, 1, 28, 28)
X_test_noisy_img = X_test_noisy.reshape(-1, 1, 28, 28)
print(X_train_img.shape)
image_loaders = make_loaders(
  X_train_noisy_img, X_train_img, X_test_noisy_img, X_test_img
)

### Le modèle

Construisez un `encoder` et un `decoder` entièrement convolutifs, puis `autoencoder`, qui les enchaîne sous ces deux noms, et `model`, l'`AutoEncoder` qui l'enveloppe. Toutes les `nn.Conv2d` ont des filtres 3 × 3 et `padding="same"` :

- `encoder`, sur des images 1 × 28 × 28 : une `nn.Conv2d` de 16 filtres (ReLU) et un `nn.MaxPool2d` 2 × 2, puis une `nn.Conv2d` de 8 filtres (ReLU) et un `nn.MaxPool2d` 2 × 2. Le code est le tenseur 8 × 7 × 7 qui en sort ;
- `decoder`, le chemin inverse : une `nn.Conv2d` de 8 filtres (ReLU) et un `nn.Upsample` de facteur 2, une `nn.Conv2d` de 16 filtres (ReLU) et un `nn.Upsample` de facteur 2, puis une dernière `nn.Conv2d` à 1 filtre suivie d'une sigmoïde, qui rend une image 1 × 28 × 28 ;
- `model`, avec la perte `nn.BCELoss()` et `input_shape=(1, 28, 28)`, la forme d'une image, pour le résumé.

Vérifiez les formes avec `ModelSummary`, et comparez son nombre de paramètres à celui du débruiteur dense de la partie 3. Pourquoi ne pas finir le décodeur par un `nn.Upsample` ?

**Documentation utile**

- [`torch.nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [`torch.nn.MaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)
- [`torch.nn.Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `nn.Conv2d(1, 16, 3, padding="same")` : 1 canal en entrée, 16 filtres 3 × 3 ; `padding="same"` garde la taille de l'image ; une petite fonction `conv(in_channels, out_channels)` qui la renvoie évite de répéter ces arguments
- `nn.MaxPool2d(2)` divise la hauteur et la largeur par 2 (28 → 14 → 7), `nn.Upsample(scale_factor=2)` les multiplie par 2 en répétant chaque pixel (7 → 14 → 28)
- la dernière couche : `conv(16, 1)`, puis `nn.Sigmoid()` ; un `nn.Upsample` en fin de réseau rendrait des blocs de 2 × 2 pixels identiques, et une sortie sans sigmoïde pourrait sortir de [0, 1]
- `print(ModelSummary(model, max_depth=2))` : la sortie de l'encodeur est `[1, 8, 7, 7]`, celle du décodeur `[1, 1, 28, 28]` (le 1 en tête est le lot de l'entrée d'exemple) ; 3,2 K paramètres en tout (3 217), contre 101 K pour le débruiteur dense de la partie 3 : chaque filtre 3 × 3 sert en tout point de l'image

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def conv(in_channels: int, out_channels: int) -> nn.Conv2d:
  return nn.Conv2d(in_channels, out_channels, 3, padding="same")


encoder = nn.Sequential(conv(1, 16), nn.ReLU(), nn.MaxPool2d(2), ..., ..., ...)
decoder = nn.Sequential(
  conv(8, 8), nn.ReLU(), nn.Upsample(scale_factor=2), ..., ..., ..., ..., ...
)
autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))
model = AutoEncoder(autoencoder, loss=..., input_shape=...)
print(ModelSummary(model, max_depth=2))
```

#### Solution

In [ ]:
def conv(in_channels: int, out_channels: int) -> nn.Conv2d:
  return nn.Conv2d(in_channels, out_channels, 3, padding="same")


encoder = nn.Sequential(
  conv(1, 16),
  nn.ReLU(),
  nn.MaxPool2d(2),  # 16 × 14 × 14
  conv(16, 8),
  nn.ReLU(),
  nn.MaxPool2d(2),  # 8 × 7 × 7, le code
)
decoder = nn.Sequential(
  conv(8, 8),
  nn.ReLU(),
  nn.Upsample(scale_factor=2),  # 8 × 14 × 14
  conv(8, 16),
  nn.ReLU(),
  nn.Upsample(scale_factor=2),  # 16 × 28 × 28
  # Lisse les blocs de l'Upsample, puis sortie dans ]0, 1[
  conv(16, 1),
  nn.Sigmoid(),
)
autoencoder = nn.Sequential(OrderedDict(encoder=encoder, decoder=decoder))
model = AutoEncoder(autoencoder, loss=nn.BCELoss(), input_shape=(1, 28, 28))
print(ModelSummary(model, max_depth=2))

### Entraînement

5 epochs suffisent pour la comparer aux débruiteurs denses, que la cellule mesure avec leur perte, l'erreur quadratique moyenne.

In [ ]:
train(model, image_loaders, max_epochs=5, name="conv")
X_test_denoised = predict(autoencoder, X_test_noisy_img)
# L'erreur quadratique moyenne, la perte des parties 2 et 3
print(f"MSE : {((X_test_denoised - X_test_img) ** 2).mean():.4f}")
show_images(X_test_noisy[:10], X_test_denoised[:10], X_test[:10], titles=titles)

### Sur des images non bruitées

Le débruiteur n'a vu que des entrées bruitées. Que rend-il d'une image propre ?

In [ ]:
X_test_rebuilt = predict(autoencoder, X_test_img[:10])
show_images(X_test[:10], X_test_rebuilt, titles=titles)

Sur une image propre, le débruiteur rend la forme du vêtement, mais lissée : il efface les détails fins, comme le logo du pull, les carreaux de la chemise ou les brides de la sandale, et force le contraste. Il a appris à garder ce qui ressemble à un vêtement et à effacer le reste ; un détail fin, à ses yeux, ressemble à du bruit. Son erreur quadratique moyenne sur les images bruitées, environ 0,021 après 5 epochs seulement, se place entre celles des deux débruiteurs denses (0,025 et 0,018), avec 3 217 paramètres au lieu de 19 612 et 101 200 : chaque filtre 3 × 3 sert en tout point de l'image, quand chaque poids d'une couche dense ne relie qu'un pixel à une unité.

## 5. Un auto-encodeur variationnel

Le notebook précédent l'a montré : décoder un code tiré au hasard ne donne pas une image vraisemblable, car les codes d'un auto-encodeur n'occupent qu'une petite partie de l'espace latent, sans forme connue. Un auto-encodeur variationnel (VAE) encode chaque image en une loi, $\mathcal{N}(\mu, \sigma^2)$, et une perte KL pousse ces lois vers $\mathcal{N}(0, I)$ : on peut alors tirer un code dans $\mathcal{N}(0, I)$ et le décoder.

Les outils de la slide « Échantillonnage et perte en PyTorch » sont fournis :

- `Sampling`, un module qui reçoit `z_mean` ($\mu$) et `z_log_var` ($\log \sigma^2$), et tire $z = \mu + \sigma \varepsilon$ avec $\varepsilon \sim \mathcal{N}(0, I)$ ;
- `VAE`, un `AutoEncoder` pour un réseau qui rend la reconstruction, `z_mean` et `z_log_var` : sa perte est celle de la reconstruction, plus la KL, avec un poids `beta` ;
- `vae_loaders`, les `DataLoader` du VAE : l'entrée et la cible sont toutes deux l'image propre.

In [ ]:
class Sampling(nn.Module):
  """z = mu + sigma * eps, avec eps ~ N(0, I) : l'astuce de reparamétrisation."""

  def forward(self, z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    eps = torch.randn_like(z_mean)
    return z_mean + torch.exp(0.5 * z_log_var) * eps  # z = μ + σ·ε


class VAE(AutoEncoder):
  """Enveloppe Lightning d'un VAE : perte de reconstruction, plus beta fois la KL."""

  def __init__(
    self,
    model: nn.Module,
    loss: nn.Module,
    beta: float = 1 / 784,
    learning_rate: float = 1e-3,
  ) -> None:
    # save_hyperparameters, dans AutoEncoder, range aussi beta dans self.hparams
    super().__init__(model, loss, learning_rate)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor], stage: str) -> torch.Tensor:
    inputs, images = batch
    reconstructions, z_mean, z_log_var = self(inputs)
    kl = -0.5 * torch.sum(1 + z_log_var - z_mean**2 - z_log_var.exp(), dim=1)
    loss = self.loss(reconstructions, images) + self.hparams.beta * kl.mean()
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    return loss


# beta = 1 / 784 : la perte BCELoss est une moyenne sur les 784 pixels,
# alors que la KL est une somme sur les dimensions du code
vae_loaders = make_loaders(X_train, X_train, X_test, X_test)

### Le modèle

Écrivez `VariationalAutoEncoder`, le réseau du VAE de la slide « Un VAE en PyTorch », avec un espace latent à `latent_dim` dimensions :

- dans `__init__`, l'encodeur : `hidden`, une `nn.Linear` de `input_dim` vers 256 unités suivie d'une ReLU, puis deux `nn.Linear` de 256 vers `latent_dim` unités sans activation, `z_mean` et `z_log_var`. Puis `sampling`, un `Sampling`, et `decoder`, un `nn.Sequential` : une `nn.Linear` de `latent_dim` vers 256 unités (ReLU), puis la sortie de `input_dim` unités en sigmoïde ;
- `encode(images)`, qui rend `z_mean` et `z_log_var` ;
- `forward(images)`, qui encode les images, tire `z` avec `sampling`, et rend la reconstruction de `z` par `decoder`, `z_mean` et `z_log_var`, ce qu'attend `VAE`.

Créez ensuite `network`, un `VariationalAutoEncoder` à `latent_dim = 2` dimensions, et `model`, le `VAE` qui l'enveloppe, avec la perte `nn.BCELoss()`, et affichez son résumé avec `ModelSummary`. La cellule après la solution l'entraîne sur les images propres, puis place les images de test dans l'espace latent avec `network.encode`.

In [ ]:
class VariationalAutoEncoder(nn.Module):
  def __init__(self, latent_dim: int) -> None:
    super().__init__()
    ...

  def encode(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """z_mean et z_log_var d'un lot d'images."""
    ...

  def forward(
    self, images: torch.Tensor
  ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Les reconstructions, z_mean et z_log_var d'un lot d'images."""
    ...


# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- dans `__init__`, chaque couche devient un attribut, comme `self.hidden = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU())`
- `self.z_mean` et `self.z_log_var` sont deux `nn.Linear(256, latent_dim)` distinctes, appliquées à la même sortie de `hidden`
- `encode` : `h = self.hidden(images)`, puis `return self.z_mean(h), self.z_log_var(h)`
- `forward` : `z = self.sampling(z_mean, z_log_var)`, puis `return self.decoder(z), z_mean, z_log_var`
- `print(ModelSummary(model, max_depth=2))` : la sortie de `sampling` est `[1, 2]`, celle de `decoder` `[1, 784]` (le 1 en tête est le lot de l'entrée d'exemple) ; 404 K paramètres en tout (404 244), presque tous dans la couche cachée de l'encodeur et dans la sortie du décodeur

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
class VariationalAutoEncoder(nn.Module):
  def __init__(self, latent_dim: int) -> None:
    super().__init__()
    self.hidden = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU())
    self.z_mean = nn.Linear(256, latent_dim)
    self.z_log_var = ...
    self.sampling = ...
    self.decoder = nn.Sequential(..., ..., ..., ...)

  def encode(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """z_mean et z_log_var d'un lot d'images."""
    h = self.hidden(images)
    return ..., ...

  def forward(
    self, images: torch.Tensor
  ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Les reconstructions, z_mean et z_log_var d'un lot d'images."""
    z_mean, z_log_var = ...
    z = ...
    return ..., z_mean, z_log_var


latent_dim = 2
network = VariationalAutoEncoder(latent_dim)
model = VAE(network, loss=...)
print(ModelSummary(model, max_depth=2))
```

#### Solution

In [ ]:
class VariationalAutoEncoder(nn.Module):
  def __init__(self, latent_dim: int) -> None:
    super().__init__()
    self.hidden = nn.Sequential(nn.Linear(input_dim, 256), nn.ReLU())
    self.z_mean = nn.Linear(256, latent_dim)
    self.z_log_var = nn.Linear(256, latent_dim)  # log σ², un réel
    self.sampling = Sampling()
    self.decoder = nn.Sequential(
      nn.Linear(latent_dim, 256),
      nn.ReLU(),
      nn.Linear(256, input_dim),
      nn.Sigmoid(),
    )

  def encode(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """z_mean et z_log_var d'un lot d'images."""
    h = self.hidden(images)
    return self.z_mean(h), self.z_log_var(h)

  def forward(
    self, images: torch.Tensor
  ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Les reconstructions, z_mean et z_log_var d'un lot d'images."""
    z_mean, z_log_var = self.encode(images)
    z = self.sampling(z_mean, z_log_var)  # z = μ + σ·ε
    return self.decoder(z), z_mean, z_log_var


latent_dim = 2
network = VariationalAutoEncoder(latent_dim)
model = VAE(network, loss=nn.BCELoss())  # + KL
print(ModelSummary(model, max_depth=2))

### Entraînement et espace latent

La cible est l'image propre elle-même. Chaque point du nuage est le $\mu$ d'une image de test, coloré selon sa classe.

In [ ]:
train(model, vae_loaders, max_epochs=20, name="vae")
with torch.no_grad():
  test_means, _ = network.encode(X_test)
plt.figure(figsize=(7, 6))
for label, name in enumerate(class_names):
  points = test_means[y_test == label]
  plt.scatter(points[:, 0], points[:, 1], s=2, alpha=0.5, label=name)
plt.legend(markerscale=5, fontsize=8)
plt.xlabel("$z_1$")
plt.ylabel("$z_2$")
plt.show()

Les μ des images de test restent presque tous entre -3 et 3 sur chaque axe, autour de l'origine : la KL tire chaque loi vers $\mathcal{N}(0, I)$. Chaque classe occupe sa région. Les chaussures (sandales, baskets, bottines) se regroupent d'un côté du nuage, séparées du reste par une bande presque vide ; les sacs ont leur région, les pantalons aussi, à côté des robes et des T-shirts. Les autres hauts (pulls, manteaux, chemises) se chevauchent : ils se ressemblent, et deux dimensions ne suffisent pas à les séparer.

## 6. Générer des images

Tirez 16 codes dans $\mathcal{N}(0, I)$, décodez-les avec `network.decoder`, et affichez les images obtenues, sur une rangée.

Ressemblent-elles à des vêtements ? Comparez-les aux images décodées depuis du bruit dans le notebook précédent.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `torch.randn(16, latent_dim, generator=torch.Generator().manual_seed(0))` : 16 codes, chaque composante tirée dans $\mathcal{N}(0, 1)$
- `predict(network.decoder, z_new)` : 16 images de 784 pixels, que `show_images` affiche

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
z_new = torch.randn(..., generator=torch.Generator().manual_seed(0))
generated = ...
show_images(generated)
```

### Solution

In [ ]:
# z ~ N(0, I)
z_new = torch.randn(16, latent_dim, generator=torch.Generator().manual_seed(0))
generated = predict(network.decoder, z_new)
show_images(generated)

La plupart ressemblent à des vêtements : des hauts, des pantalons, des chaussures, flous comme toutes les images d'un VAE à deux dimensions. Quelques-unes, pâles, mélangent deux sortes de vêtements : leur code est tombé entre deux régions, où le décodeur hésite. Dans le notebook précédent, un code tiré au hasard ne donnait que des taches : ici, la KL a placé les codes des images d'entraînement là où $\mathcal{N}(0, I)$ tire les siens, et le décodeur sait les décoder. Un espace latent plus grand donnerait des images plus nettes, mais on ne pourrait plus en dessiner la carte.

### La carte de l'espace latent

Avec 2 dimensions, on peut décoder une grille régulière de codes, de -2 à 2 sur chaque axe, là où $\mathcal{N}(0, I)$ tire presque tous ses points : chaque case montre ce que le décodeur rend en ce point. Comparez avec le nuage de la partie 5.

In [ ]:
n = 12
grid = torch.linspace(-2, 2, n).tolist()
# z_2 décroît de haut en bas, comme l'axe vertical du nuage
codes = torch.tensor([[z1, z2] for z2 in grid[::-1] for z1 in grid])
images = predict(network.decoder, codes).reshape(n, n, 28, 28)
plt.figure(figsize=(8, 8))
plt.imshow(images.permute(0, 2, 1, 3).reshape(n * 28, n * 28), cmap="gray_r")
plt.axis("off")
plt.show()